# BombermanAI Round Robin Benchmark

This Kaggle notebook is used to:
- clone the repo from GitHub
- install dependencies
- run the same seed set for every agent
- compare two environments: normal and fog of war
- export detailed tables for the report
- record decision time per step for speed comparison

Notes:
- Replace `REPO_URL` with your real GitHub link.
- The benchmark here is a round-robin style comparison over the same seed range so the results are fair and reproducible.


In [ ]:
# 1) Clone repository
import os
import sys
import math
import json
import statistics
from pathlib import Path

REPO_URL = "https://github.com/<your-user-or-org>/Boomber-AI-game.git"
REPO_DIR = "Boomber-AI-game"

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# 2) Install dependencies if needed
import sys

!{sys.executable} -m pip install -q -r requirements.txt

In [ ]:
# 3) Import benchmark helpers and agent factories
import pandas as pd

from benchmarks.benchmark import run_game
from agents.random_agent import RandomAgent
from agents.bfs_agent import BFSAgent
from agents.dfs_agent import DFSAgent
from agents.greedy_agent import GreedyAgent
from agents.astar_agent import AStarAgent
from agents.hill_climbing_agent import HillClimbingAgent
from agents.simulated_annealing_agent import SimulatedAnnealingAgent
from agents.backtracking_agent import BacktrackingAgent
from agents.min_conflicts_agent import MinConflictsAgent
from agents.online_search_agent import OnlineSearchAgent
from agents.and_or_search_agent import AndOrSearchAgent
from agents.minimax_agent import MinimaxAgent
from agents.expectimax_agent import ExpectimaxAgent


AGENTS = [
    ("Random", lambda: RandomAgent(seed=42)),
    ("BFS", lambda: BFSAgent()),
    ("DFS", lambda: DFSAgent()),
    ("Greedy", lambda: GreedyAgent()),
    ("A*", lambda: AStarAgent()),
    ("HillClimbing", lambda: HillClimbingAgent()),
    ("SimulatedAnnealing", lambda: SimulatedAnnealingAgent()),
    ("Backtracking", lambda: BacktrackingAgent()),
    ("MinConflicts", lambda: MinConflictsAgent()),
    ("OnlineSearch", lambda: OnlineSearchAgent()),
    ("AndOrSearch", lambda: AndOrSearchAgent()),
    ("Minimax", lambda: MinimaxAgent(depth=2)),
    ("Expectimax", lambda: ExpectimaxAgent(depth=2)),
]

print("Agents loaded:", [name for name, _ in AGENTS])

In [ ]:
# 4) Benchmark runner for one environment mode (multi-core)
import os
import math
from concurrent.futures import ProcessPoolExecutor


def _run_single_game(args):
    agent_name, agent_factory, seed, fog_of_war, fow_radius = args
    agent = agent_factory()
    result = run_game(
        agent,
        seed=seed,
        render_mode=False,
        fog_of_war=fog_of_war,
        fow_radius=fow_radius,
    )
    decision_times = result.get("decision_times_ms", [])
    return {
        "seed": seed,
        "score": result["score"],
        "steps": result["steps"],
        "won": result["won"],
        "survival_time": float(result["steps"]),
        "avg_decision_time_ms": float(result.get("avg_decision_time_ms", 0.0)),
        "max_decision_time_ms": float(result.get("max_decision_time_ms", 0.0)),
        "p95_decision_time_ms": float(result.get("p95_decision_time_ms", 0.0)),
        "timeout_rate_100ms": float(sum(1 for t in decision_times if t > 100.0) / len(decision_times)) if decision_times else 0.0,
    }


def run_agent_on_seeds(agent_name, agent_factory, *, num_games, start_seed, fog_of_war, fow_radius, workers=None):
    workers = workers or max(1, os.cpu_count() or 1)
    tasks = [(agent_name, agent_factory, start_seed + idx, fog_of_war, fow_radius) for idx in range(num_games)]
    rows = []
    with ProcessPoolExecutor(max_workers=workers) as executor:
        for result in executor.map(_run_single_game, tasks, chunksize=max(1, math.ceil(len(tasks) / (workers * 4)))):
            rows.append(result)
    df = pd.DataFrame(rows)
    df.insert(0, "agent", agent_name)
    df.insert(1, "mode", "fow" if fog_of_war else "normal")
    return df


def summarize_frame(df):
    if df.empty:
        return {}
    win_rate = float(df["won"].mean())
    score = df["score"]
    steps = df["steps"]
    return {
        "games": int(len(df)),
        "win_rate": win_rate,
        "average_score": float(score.mean()),
        "median_score": float(score.median()),
        "score_std": float(score.std(ddof=0)) if len(score) > 1 else 0.0,
        "min_score": float(score.min()),
        "max_score": float(score.max()),
        "average_steps": float(steps.mean()),
        "median_steps": float(steps.median()),
        "steps_std": float(steps.std(ddof=0)) if len(steps) > 1 else 0.0,
        "min_steps": float(steps.min()),
        "max_steps": float(steps.max()),
        "avg_decision_time_ms": float(df["avg_decision_time_ms"].mean()),
        "max_decision_time_ms": float(df["max_decision_time_ms"].max()),
        "p95_decision_time_ms": float(df["p95_decision_time_ms"].mean()),
        "timeout_rate_100ms": float(df["timeout_rate_100ms"].mean()),
    }


def benchmark_suite(*, num_games=1000, start_seed=42, fow_radius=4, workers=None):
    workers = workers or max(1, os.cpu_count() or 1)
    print(f"Using {workers} worker processes (os.cpu_count={os.cpu_count()})")
    all_rows = []
    summaries = []

    for mode_name, fog_of_war in [("normal", False), ("fow", True)]:
        for agent_name, factory in AGENTS:
            print(f"Running {agent_name} | mode={mode_name}")
            df = run_agent_on_seeds(
                agent_name,
                factory,
                num_games=num_games,
                start_seed=start_seed,
                fog_of_war=fog_of_war,
                fow_radius=fow_radius,
                workers=workers,
            )
            all_rows.append(df)

            summary = summarize_frame(df)
            summary.update({"agent": agent_name, "mode": mode_name})
            summaries.append(summary)

    results = pd.concat(all_rows, ignore_index=True)
    summary_df = pd.DataFrame(summaries)
    return results, summary_df


NUM_GAMES = 1000
START_SEED = 42
FOW_RADIUS = 4
WORKERS = max(1, os.cpu_count() or 1)

# Change this to a smaller value for a smoke test if needed:
# NUM_GAMES = 20
# WORKERS = 2

results_df, summary_df = benchmark_suite(
    num_games=NUM_GAMES,
    start_seed=START_SEED,
    fow_radius=FOW_RADIUS,
    workers=WORKERS,
)

summary_df


In [ ]:
# 5) Detailed analysis tables
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

summary_df = summary_df.sort_values(["mode", "win_rate", "average_score"], ascending=[True, False, False]).reset_index(drop=True)
summary_df

In [ ]:
# 6) Compare normal vs fog of war side by side
pivot = summary_df.pivot(index="agent", columns="mode", values=["win_rate", "average_score", "average_steps", "median_score", "score_std", "avg_decision_time_ms", "max_decision_time_ms", "p95_decision_time_ms", "timeout_rate_100ms"])
pivot


In [ ]:
# 7) Export artifacts for report writing
output_dir = Path("benchmark_outputs")
output_dir.mkdir(exist_ok=True)

results_path = output_dir / "round_robin_raw_results.csv"
summary_path = output_dir / "round_robin_summary.csv"
pivot_path = output_dir / "round_robin_pivot.csv"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
pivot.to_csv(pivot_path)

print("Saved:")
print(results_path)
print(summary_path)
print(pivot_path)

## Reporting tips

In the experimental section, you can report:
- number of games: `1000` per mode
- two modes: `normal` and `fog of war`
- main metrics: `win rate`, `average score`, `average steps`, `median score`, `score std`
- latency metrics: `avg_decision_time_ms`, `p95_decision_time_ms`, `max_decision_time_ms`, `timeout_rate_100ms`
- use the same seeds to keep the comparison fair
- set `WORKERS = os.cpu_count()` to use all available CPU cores in Kaggle

If needed, you can also add:
- boxplots for score distribution
- top-k and bottom-k seeds for each agent
- stability comparison between normal and fog of war
